In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

IN = Path("outputs/hypo3_internal/int_w.csv")
OUT = Path("outputs/hypo3_internal_reg")
PIC = Path("output_pic/hypo3_internal_reg")

OUT.mkdir(parents=True, exist_ok=True)
PIC.mkdir(parents=True, exist_ok=True)

assert IN.exists(), "Run Hypo3_internal.ipynb first: outputs/hypo3_internal/int_w.csv not found."

w = pd.read_csv(IN, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
w.head()


,date,gpu_ret,gpu_ret_n,gpu_act_n,gpu_in_n,gpu_out_n,gpu_match_n,gpu_in_sh,gpu_out_sh,gpu_match_sh,...,ram_match_sh,ram_med_px,ram_logp_iqr,ram_px_cv,ram_ret_iqr,ram_ret_abs,ram_ddr5_sh,ram_cap_avg,ram_spd_avg,ram_cap_hi_sh
0,2024-03-03,NaN,0,727,727,0,0,1.000000,NaN,0.000000,...,0.000000,108900.0,1.442339,1.073842,NaN,NaN,0.298851,28.620690,3948.831418,0.505747
1,2024-03-10,0.002189,646,760,40,7,720,0.052632,0.009629,0.947368,...,0.747335,110020.0,1.481910,1.064067,0.000000,0.00000,0.299574,28.765458,3954.711087,0.505330
2,2024-03-17,0.003092,655,750,18,28,732,0.024000,0.036842,0.976000,...,0.806519,113400.0,1.512188,1.050350,0.000107,0.00005,0.313354,29.449001,3986.031546,0.518402
3,2024-03-24,0.002050,648,747,21,24,726,0.028112,0.032000,0.971888,...,0.795383,114000.0,1.494786,1.050172,0.000412,0.00000,0.317943,29.794334,3995.579224,0.522560
4,2024-03-31,0.001093,652,760,31,18,729,0.040789,0.024096,0.959211,...,0.801494,114640.0,1.512351,1.044889,0.000284,0.00000,0.316969,29.553895,4002.056564,0.520811


In [2]:

P = ["gpu", "cpu", "ram"]

reg = pd.DataFrame({"date": w["date"]})


def pos(p):
    return w[f"{p}_act_n"] > 0


def ok(p):
    return pos(p) & pos(p).shift(1, fill_value=False)


def lchg(s, m):
    x = s.where(s > 0)
    y = np.log(x).diff()
    return y.where(m)


def dchg(s, m):
    return s.diff().where(m)


def nsh(p):
    return w[f"{p}_in_sh"].where(ok(p))


def lag1(s, m):
    return s.shift(1).where(m)

for p in P:
    m = ok(p)
    reg[f"{p}_ret"] = w[f"{p}_ret"].where(pos(p))
    reg[f"{p}_n_chg"] = lchg(w[f"{p}_act_n"], m)
    reg[f"{p}_new_sh"] = nsh(p)
    reg[f"{p}_riq_l1"] = lag1(w[f"{p}_ret_iqr"], m)

reg["gpu_high_chg"] = dchg(w["gpu_high_sh"], ok("gpu"))
reg["gpu_vram_chg"] = dchg(w["gpu_vram_avg"], ok("gpu"))
reg["cpu_amd_chg"] = dchg(w["cpu_amd_sh"], ok("cpu"))
reg["ram_ddr5_chg"] = dchg(w["ram_ddr5_sh"], ok("ram"))
reg["ram_cap_chg"] = dchg(w["ram_cap_avg"], ok("ram"))

cols = [c for c in reg.columns if c != "date"]
raw_n = len(reg)
reg.head()


,date,gpu_ret,gpu_n_chg,gpu_new_sh,gpu_riq_l1,cpu_ret,cpu_n_chg,cpu_new_sh,cpu_riq_l1,ram_ret,ram_n_chg,ram_new_sh,ram_riq_l1,gpu_high_chg,gpu_vram_chg,cpu_amd_chg,ram_ddr5_chg,ram_cap_chg
0,2024-03-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-03-10,0.002189,0.044392,0.052632,NaN,-0.001478,0.023670,0.029240,NaN,0.002896,0.180617,0.252665,NaN,-0.000927,-0.036723,-0.009245,0.000723,0.144769
2,2024-03-17,0.003092,-0.013245,0.024000,0.000201,0.000508,-0.005865,0.035294,0.000561,0.001473,0.013764,0.193481,0.000000,0.001404,-0.027193,0.002270,0.013781,0.683543
3,2024-03-24,0.002050,-0.004008,0.028112,0.002399,0.000673,0.000000,0.029412,0.000103,-0.003016,0.002101,0.204617,0.000107,0.003373,-0.035074,-0.005882,0.004589,0.345333
4,2024-03-31,0.001093,0.017253,0.040789,0.001316,0.001044,0.005865,0.052632,0.000000,-0.001508,-0.016932,0.198506,0.000412,-0.008724,-0.008786,-0.002236,-0.000974,-0.240438


In [3]:

why = {
    "ret": "first/gap return unavailable",
    "n_chg": "log-difference requires current and previous positive active count",
    "new_sh": "entry share requires previous positive active count; post-gap value is artifact",
    "riq_l1": "lagged internal-volatility variable is undefined after first/gap week",
    "mix": "first-difference requires current and previous valid mix share",
}

rows = []
for c in cols:
    if c.endswith("_ret"):
        typ = "ret"
    elif c.endswith("_n_chg"):
        typ = "n_chg"
    elif c.endswith("_new_sh"):
        typ = "new_sh"
    elif c.endswith("_riq_l1"):
        typ = "riq_l1"
    else:
        typ = "mix"
    rows.append({
        "var": c,
        "na": int(reg[c].isna().sum()),
        "na_pct": round(reg[c].isna().mean() * 100, 2),
        "mech": "structural",
        "method": "drop",
        "reason": why[typ],
    })

miss = pd.DataFrame(rows)
reg2 = reg.dropna(subset=cols).reset_index(drop=True)

miss.loc[len(miss)] = {
    "var": "final_rows",
    "na": raw_n - len(reg2),
    "na_pct": round((raw_n - len(reg2)) / raw_n * 100, 2),
    "mech": "structural_complete_case",
    "method": "drop",
    "reason": f"kept {len(reg2)} of {raw_n} weeks after structural-cleaning",
}

reg2.to_csv(OUT / "int_reg.csv", index=False)
miss.to_csv(OUT / "miss.csv", index=False)

print("saved:", OUT / "int_reg.csv")
print("saved:", OUT / "miss.csv")
print("shape:", reg2.shape)
miss


saved: outputs\hypo3_internal_reg\int_reg.csv
saved: outputs\hypo3_internal_reg\miss.csv
shape: (103, 18)


,var,na,na_pct,mech,method,reason
0,gpu_ret,5,4.55,structural,drop,first/gap return unavailable
1,gpu_n_chg,5,4.55,structural,drop,log-difference requires current and previous p...
2,gpu_new_sh,5,4.55,structural,drop,entry share requires previous positive active ...
3,gpu_riq_l1,7,6.36,structural,drop,lagged internal-volatility variable is undefin...
4,cpu_ret,5,4.55,structural,drop,first/gap return unavailable
5,cpu_n_chg,5,4.55,structural,drop,log-difference requires current and previous p...
6,cpu_new_sh,5,4.55,structural,drop,entry share requires previous positive active ...
7,cpu_riq_l1,7,6.36,structural,drop,lagged internal-volatility variable is undefin...
8,ram_ret,5,4.55,structural,drop,first/gap return unavailable
9,ram_n_chg,5,4.55,structural,drop,log-difference requires current and previous p...


In [4]:

corr = reg2[cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, vmin=-1, vmax=1)
ax.set_xticks(range(len(cols)))
ax.set_yticks(range(len(cols)))
ax.set_xticklabels(cols, rotation=90)
ax.set_yticklabels(cols)
fig.colorbar(im, ax=ax, label="corr")
ax.set_title("internal regression variables")
plt.tight_layout()
plt.savefig(PIC / "01_corr.png", dpi=160)
plt.close()

print("saved:", PIC / "01_corr.png")


saved: output_pic\hypo3_internal_reg\01_corr.png
